# SadTalker on a free Colab GPU

This notebook runs SadTalker interactively on a Colab GPU when one is available. Select **Runtime → Change runtime type → T4 GPU** (or another GPU), then run the cells in order. Free GPU access is not guaranteed and sessions may disconnect or stop; this is not an unattended GitHub Actions runner.

The image and audio you upload are sent to the Google Colab runtime for processing. This notebook does not upload them to this GitHub repository or use a paid API. Download the MP4 before the runtime ends. SadTalker animates lip-sync and some face/head movement only; it does not create body/hand gestures or walking from a still image. The 1920×1080 output is upscaled from SadTalker's 512-pixel model, so it is not native 1080p detail.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is attached. In Colab choose Runtime > Change runtime type > GPU, reconnect, then rerun this cell.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
# Install the legacy Python/PyTorch versions supported by this SadTalker checkout.
import os
import subprocess
from pathlib import Path

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv", "ffmpeg", "libgl1"], check=True)

repo = Path("/content/SadTalker")
venv_python = "/content/sadtalker-venv/bin/python"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "feature/github-actions-video-pipeline", "https://github.com/prakashdeepika/SadTalker.git", str(repo)], check=True)
subprocess.run(["python3.10", "-m", "venv", "/content/sadtalker-venv"], check=True)

def run(command, **kwargs):
    subprocess.run(command, check=True, **kwargs)

run([venv_python, "-m", "pip", "install", "--upgrade", "pip<25"])
run([
    venv_python, "-m", "pip", "install",
    "torch==1.12.1+cu113", "torchvision==0.13.1+cu113", "torchaudio==0.12.1",
    "--extra-index-url", "https://download.pytorch.org/whl/cu113",
])
run([venv_python, "-m", "pip", "install", "-r", str(repo / "requirements.txt")])
print("Environment installed at", venv_python)

In [ ]:
# Download SadTalker's 512 model and the face-enhancement weights.
import subprocess

downloads = {
    "checkpoints/mapping_00109-model.pth.tar": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar",
    "checkpoints/SadTalker_V0.0.2_512.safetensors": "https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_512.safetensors",
    "gfpgan/weights/alignment_WFLW_4HG.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
    "gfpgan/weights/detection_Resnet50_Final.pth": "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
    "gfpgan/weights/GFPGANv1.4.pth": "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth",
    "gfpgan/weights/parsing_parsenet.pth": "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth",
}

for relative_path, url in downloads.items():
    destination = repo / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["curl", "--fail", "--location", "--retry", "3", url, "--output", str(destination)], check=True)

In [ ]:
# Upload exactly one source image and one audio file from your computer.
from google.colab import files
from pathlib import Path
import shutil

uploaded = files.upload()
image_files = [name for name in uploaded if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}]
audio_files = [name for name in uploaded if Path(name).suffix.lower() in {".mp3", ".wav", ".m4a", ".flac", ".ogg"}]
if len(image_files) != 1 or len(audio_files) != 1:
    raise ValueError("Upload exactly one image (PNG/JPG/WEBP) and one audio file (MP3/WAV/M4A/FLAC/OGG).")

work = Path("/content/work")
work.mkdir(parents=True, exist_ok=True)
source_image = work / Path(image_files[0]).name
source_audio = work / Path(audio_files[0]).name
shutil.move(image_files[0], source_image)
shutil.move(audio_files[0], source_audio)
normalized_audio = work / "audio-16k-mono.wav"
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(source_audio),
    "-vn", "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", str(normalized_audio),
], check=True)
if not normalized_audio.is_file() or normalized_audio.stat().st_size == 0:
    raise RuntimeError("Audio conversion did not create a valid WAV file.")
print("Image:", source_image.name, "| audio:", source_audio.name)

In [ ]:
# Generate the full-frame talking-head video on the selected GPU.
import subprocess

intermediate = work / "sadtalker-intermediate.mp4"
subprocess.run([
    venv_python, str(repo / "scripts/generate_video.py"),
    "--source-image", str(source_image),
    "--audio", str(normalized_audio),
    "--output", str(intermediate),
    "--checkpoint-dir", str(repo / "checkpoints"),
    "--preprocess", "full",
    "--size", "512",
    "--enhancer", "gfpgan",
    "--expression-scale", "1.2",
], cwd=str(repo), check=True)

output_video = work / "sadtalker-1920x1080.mp4"
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(intermediate), "-i", str(source_audio),
    "-map", "0:v:0", "-map", "1:a:0",
    "-vf", "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,setsar=1",
    "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p", "-r", "25",
    "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart", str(output_video),
], check=True)

probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "stream=codec_type,width,height",
    "-of", "default=noprint_wrappers=1", str(output_video),
], check=True, capture_output=True, text=True)
print(probe.stdout)
if "width=1920" not in probe.stdout or "height=1080" not in probe.stdout or "codec_type=audio" not in probe.stdout:
    raise RuntimeError("Output validation failed: expected 1920x1080 video and an audio stream.")

In [ ]:
from IPython.display import Video, display
display(Video(str(output_video), embed=True))
files.download(str(output_video))